# Progetto DIA - A.A 2025/26

Autori: Justin Carideo (justin.carideo@studio.unibo.it), Laura Bertozzi ()

# Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sklearn

Importiamo i dataset presi dal sito https://www.kaggle.com. Questi due dataset fanno riferimento a dati in ambito agricolo, in particolare:
1) **Soil Nutrients** -> dataset incentrato su aspetti ambientali per individuare il tipo di coltura dato il tipo di terreno e nutrienti presenti (https://www.kaggle.com/datasets/snmahsa/soil-nutrients)
2) **Crop and Soil DataSet** -> dataset incentrato sullo stesso campo ma che vuole individuare il tipo di fertilizzante dato il tipo di terreno e la coltura presente. (https://www.kaggle.com/datasets/shankarpriya2913/crop-and-soil-dataset)

In particolare l'idea è quello di creare un modello di **classificazione**(multiclasse) che prima preveda di individuare il tipo di coltura adatta per un tipo di terreno e successivamente individuare anche il tipo fertilizzante per quel tipo di coltura in quel determinato terreno.

In [2]:
import kaggle
import os
import os.path

SOIL_DATASET_ID = "snmahsa/soil-nutrients"
CROP_AND_SOIL_DATASET_ID = "shankarpriya2913/crop-and-soil-dataset"
SOIL_DATASET_PATH = "./data/Soil Nutrients.csv"
CROP_AND_SOIL_DATASET_PATH = "./data/data_core.csv"


download_dir = "./data"
os.makedirs(download_dir, exist_ok=True) # crea la directory se non esiste

if not os.path.exists(CROP_AND_SOIL_DATASET_ID):
    kaggle.api.dataset_download_files(
        CROP_AND_SOIL_DATASET_ID,
        path=download_dir,
        unzip=True
    )

if not os.path.exists(SOIL_DATASET_PATH):
    kaggle.api.dataset_download_files(
        SOIL_DATASET_ID,
        path=download_dir,
        unzip=True
    )

Dataset URL: https://www.kaggle.com/datasets/shankarpriya2913/crop-and-soil-dataset


Creiamo i dataframe di entrambi i dataset:

In [3]:
soil_df = pd.read_csv(SOIL_DATASET_PATH)
crop_df = pd.read_csv(CROP_AND_SOIL_DATASET_PATH)

Ora che abbiamo ottenuto i dataset dobbiamo decidere che modello attuare su questi dataset e visto che non hanno indici univoci per riga possiamo attuare una **classificazione** per la predizione di colture e fertilizzanti. In particolare potremmo ragionare con una sorta di *pipeline* dove i dati di entrambi vengono utilizzati per ottenere una classificazione della coltura desiderata date le informazioni relative al terreno e utilizzare i dati ottenuti per calcolare anche il tipo di fertilizzante da attuare.

Questa è una analisi a priori, una volta svolta l'*analisi esplorativa* si potrà procedere con la costruzione del modello.

# Analisi Esplorativa

In [6]:
soil_df

,Name,Fertility,Photoperiod,Temperature,Rainfall,pH,Light_Hours,Light_Intensity,Rh,Nitrogen,Phosphorus,Potassium,Yield,Category_pH,Soil_Type,Season,N_Ratio,P_Ratio,K_Ratio
0,Strawberry,Moderate,Day Neutral,20.887923,747.860765,6.571548,13.091483,533.762876,91.197196,170.800381,118.670058,243.331211,20.369555,low_acidic,Loam,Summer,10.0,10.0,10.0
1,Strawberry,Moderate,Day Neutral,18.062721,711.104329,6.251806,13.063016,505.789101,91.939623,179.290364,121.020244,246.910378,20.402751,low_acidic,Loam,Spring,10.0,10.0,10.0
2,Strawberry,Moderate,Short Day Period,16.776782,774.038247,6.346916,12.945927,512.985617,91.387286,181.440732,116.936806,242.699601,19.158847,low_acidic,Loam,Summer,10.0,10.0,10.0
3,Strawberry,Moderate,Short Day Period,14.281000,665.633506,6.259598,13.318922,484.860067,91.254598,176.165282,122.233153,237.096892,20.265745,low_acidic,Loam,Summer,10.0,10.0,10.0
4,Strawberry,Moderate,Day Neutral,21.444490,806.531455,6.384368,13.312915,512.747307,92.354829,182.935334,126.088234,243.880364,20.397336,low_acidic,Loam,Spring,10.0,10.0,10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15395,Green Peas,Moderate,Short Day Period,18.237489,1079.572958,6.777649,6.913269,314.935840,65.057374,150.515721,48.860186,124.688035,4.977256,neutral,Sandy,Fall,5.0,10.0,10.0
15396,Green Peas,Moderate,Short Day Period,16.603638,958.201820,5.839441,6.829060,345.860296,66.747340,144.310767,44.647790,121.589160,4.987133,neutral,Sandy,Fall,5.0,10.0,10.0
15397,Green Peas,Moderate,Short Day Period,12.154144,947.899222,6.499094,6.938902,320.293737,65.803531,147.068405,42.351771,120.392912,5.043142,low_acidic,Sandy,Fall,5.0,10.0,10.0
15398,Green Peas,Moderate,Short Day Period,17.493029,863.902923,5.940159,6.778806,300.501265,64.563183,144.416616,44.405726,119.291683,4.687349,low_acidic,Sandy,Spring,5.0,10.0,10.0


In [7]:
crop_df

,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,26.00,52.00,38.00,Sandy,Maize,37,0,0,Urea
1,29.00,52.00,45.00,Loamy,Sugarcane,12,0,36,DAP
2,34.00,65.00,62.00,Black,Cotton,7,9,30,14-35-14
3,32.00,62.00,34.00,Red,Tobacco,22,0,20,28-28
4,28.00,54.00,46.00,Clayey,Paddy,35,0,0,Urea
...,...,...,...,...,...,...,...,...,...
7995,35.30,59.61,44.25,Loamy,Oil seeds,10,14,10,Urea
7996,39.39,71.67,49.34,Black,Barley,35,0,0,10-26-26
7997,35.79,67.64,45.04,Red,Barley,41,0,0,Urea
7998,37.78,73.38,36.03,Black,Tobacco,10,3,30,DAP


In [4]:
soil_df.describe()

,Temperature,Rainfall,pH,Light_Hours,Light_Intensity,Rh,Nitrogen,Phosphorus,Potassium,Yield,N_Ratio,P_Ratio,K_Ratio
count,15400.000000,15400.000000,15400.000000,15400.000000,15400.000000,15400.000000,15400.000000,15400.000000,15400.000000,15400.000000,15400.000000,15400.000000,15400.000000
mean,20.801671,948.814222,6.473372,9.459365,398.048832,67.117251,142.769483,107.659893,180.481105,22.749990,12.636364,11.704545,12.477273
std,4.415164,340.884493,0.449111,2.588466,190.512173,19.005233,58.524592,72.778284,103.989340,15.541414,14.028173,5.761853,7.805376
min,9.355908,409.927161,4.888871,5.044913,69.146572,29.877266,41.725552,13.155191,34.982329,0.770213,5.000000,10.000000,5.000000
25%,17.915142,707.799164,6.245973,7.017129,265.438161,53.009874,117.631453,57.627430,107.263425,11.978009,10.000000,10.000000,10.000000
50%,19.929578,846.623453,6.466286,8.985917,385.299681,63.248496,140.524941,87.731849,163.891242,21.232615,10.000000,10.000000,10.000000
75%,22.936714,1109.157262,6.697254,12.019180,508.211962,89.996774,160.186895,145.298926,223.236991,28.812111,10.000000,10.000000,10.000000
max,39.913892,2508.052849,8.023210,15.918024,985.186247,99.927439,409.639573,360.043619,579.953931,66.619242,75.000000,37.500000,37.500000


In [5]:
crop_df.describe()

,Temparature,Humidity,Moisture,Nitrogen,Potassium,Phosphorous
count,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000
mean,30.338895,59.210731,43.580862,18.429125,3.916375,18.512500
std,4.478262,8.177366,12.596156,11.852406,5.494807,13.244113
min,20.000000,40.020000,20.000000,0.000000,0.000000,0.000000
25%,27.050000,53.277500,33.967500,9.000000,0.000000,8.000000
50%,30.240000,59.110000,42.250000,14.000000,1.000000,18.000000
75%,33.460000,65.082500,52.950000,26.000000,5.000000,30.000000
max,40.000000,80.000000,70.000000,46.000000,23.000000,46.000000


La parte più importante a quanto pare sono le colture perché non hanno una corrispondenza biunivoca, di conseguenza dovremmo capire come mappare i dati tra i due dataset.

In [11]:
soil_df["Name"].unique()

<StringArray>
[    'Strawberry',     'Watermelon',         'Grapes',        'Arugula',
           'Beet',          'Chard',          'Cress',         'Endive',
           'Kale',        'Lettuce',      'Radicchio',        'Spinach',
       'Tomatoes',      'Eggplants',      'Asparagus', 'Chilli Peppers',
        'Cabbage',      'Cucumbers',       'Potatoes',   'Cauliflowers',
       'Broccoli',     'Green Peas']
Length: 22, dtype: str

In [13]:
crop_df["Crop Type"].unique()

<StringArray>
[      'Maize',   'Sugarcane',      'Cotton',     'Tobacco',       'Paddy',
      'Barley',       'Wheat',     'Millets',   'Oil seeds',      'Pulses',
 'Ground Nuts']
Length: 11, dtype: str

Notiamo che i due dataset non hanno nessuna corrispondenza a livello di colture coltivate e in alcuni casi abbiamo scale diverse(`Nitrogen`, `Potassium`,...)